In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 48.1 MB/s eta 0:00:00


In [3]:
import gradio as gr
import tensorflow as tf
import torch
import lightning as L
from torchvision import transforms
import timm
import torch.nn as nn
import numpy as np
from PIL import Image

class ViTFruitClassifier(L.LightningModule):
    def __init__(self, num_classes=5, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model('vit_base_patch16_224', pretrained=False)
        self.model.head = nn.Linear(self.model.head.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

keras_path = "/content/drive/MyDrive/deeplearning pj/cnn_test_3/cnn3.keras"
ckpt_path = "/content/drive/MyDrive/deeplearning pj/checkpoint/vit.ckpt"

# Load Keras
try:
    keras_model = tf.keras.models.load_model(keras_path)
    print("Đã load Keras Model")
except:
    keras_model = None
    print(" Không tìm thấy file Keras")

# Load PyTorch
try:
    pytorch_model = ViTFruitClassifier.load_from_checkpoint(ckpt_path, map_location=torch.device('cpu'))
    pytorch_model.eval()
    print("Đã load PyTorch Model")
except:
    pytorch_model = None
    print("Không tìm thấy file PyTorch CKPT (Check đường dẫn)")

def predict_image(image, model_choice):
    if image is None:
        return None

    labels = ["Apple", "Banana", "Grape", "Mango", "Strawberry"]

    if model_choice == "Keras (CNN)":
        if keras_model is None: return {"Lỗi": "Chưa load được model Keras"}

        # Xử lý ảnh cho Keras
        img = image.resize((128, 128)) # Size model Keras
        img_arr = np.array(img)
        img_arr = np.expand_dims(img_arr, axis=0)

        # Dự đoán
        pred = keras_model.predict(img_arr)[0]
        return {labels[i]: float(pred[i]) for i in range(len(labels))}

    else: # PyTorch
        if pytorch_model is None: return {"Lỗi": "Chưa load được model PyTorch"}

        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])
        img_tensor = transform(image).unsqueeze(0)


        with torch.no_grad():
            logits = pytorch_model(img_tensor)
            probs = torch.nn.functional.softmax(logits, dim=1)[0]

        return {labels[i]: float(probs[i]) for i in range(len(labels))}


demo = gr.Interface(
    fn=predict_image,
    inputs=[
        gr.Image(type="pil", label="Tải ảnh hoa quả lên"),
        gr.Radio(["Keras (CNN)", "PyTorch (ViT)"], label="Chọn Model", value="PyTorch (ViT)")
    ],
    outputs=gr.Label(num_top_classes=3, label="Kết quả dự đoán"),
    title="Demo Phân loại Hoa Quả",
    description="So sánh trực tiếp mô hình CNN (Keras) và ViT (PyTorch Lightning)."
)

demo.launch(share=True, debug=True)

Đã load Keras Model
Đã load PyTorch Model
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6eb8c7652ae0b56d58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 523ms/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6eb8c7652ae0b56d58.gradio.live
